# 05. 리텐션 캠페인 A/B 사전 설계

실제 실험 결과가 아닌 공개 로그 기반 설계다. **3/31 UTC 하루 종료까지**의 활동·구매 이력으로 Recency 15–29일 위험군을 분류하고, **4/1–4/30**의 활동으로 참고 baseline을 계산한다.

먼저 저장소 루트에서 `python -m src.ab_design`을 실행한다. 원본 6개월의 3개 컬럼을 50만 행씩 읽고 사전 구매 이력 캐시·집계 CSV·설계서 표를 생성한다. 기존 3/31 recency와 결과 라벨은 재사용한다.

In [1]:
import sys
from pathlib import Path
root = Path.cwd() if (Path.cwd() / "config.py").exists() else Path.cwd().parent
sys.path.insert(0, str(root))
import config
import pandas as pd
from IPython.display import Markdown, display
from src.ab_design import load_risk_population, summarize_risk, summary_markdown

risk = load_risk_population()
summary = summarize_risk(risk)
display(Markdown(summary_markdown(summary)))

| 구분 | 대상자 수 | 4월 재방문자 수 | baseline | 설계 대립값(+5%p) | 군당 표본 |
|---|---:|---:|---:|---:|---:|
| 전체 위험군 | 2,200,340 | 571,010 | 25.95% | 30.95% | 1,276 |
| 기준일 이전 구매 경험군 | 322,372 | 136,385 | 42.31% | 47.31% | 1,552 |
| 기준일 이전 비구매군 | 1,877,968 | 434,625 | 23.14% | 28.14% | 1,195 |

## 입력 시점과 층화

`is_buyer_at_reference`는 관측 시작일부터 3/31 종료까지 구매가 있었는지다. 04의 `is_buyer`는 4월을 포함하므로 이 계산에 사용하지 않는다. 구매 여부만으로 고객을 고가치·저가치라 부르지 않는다.

31.5% vs 10.3%는 사후 구매 구성비이며 사전 예측력·인과효과가 아니다. 위 표의 층별 baseline을 사용한다.

In [2]:
stored = pd.read_csv(config.REPORTS_DIR / "ab_design_summary.csv")
pd.testing.assert_frame_equal(summary, stored, check_exact=False, rtol=1e-12)
assert summary.iloc[1:]["users"].sum() == summary.iloc[0]["users"]
assert summary.iloc[1:]["returned_users"].sum() == summary.iloc[0]["returned_users"]
assert risk["status"].isin(["retained", "churned"]).all()
assert risk["recency_at_ref"].between(15, 29).all()
print("저장 CSV 일치 · 층별 인원/재방문자 합계 일치 · 위험군/라벨 검사 통과")

저장 CSV 일치 · 층별 인원/재방문자 합계 일치 · 위험군/라벨 검사 통과


## 표본과 실행 설계

alpha 0.05, power 0.80, 절대 MDE +5%p, 양측 검정, 처치·대조 1:1이다. 표본 수는 올림하며 설계서 표와 같은 함수를 쓴다.

전체를 1차 판단, 층별을 탐색 분석으로 둔다. 대상 풀의 층 비중을 유지해 추출한 뒤 각 층 내에서 1:1 배정한다. 전체 표본 확보는 각 층의 검정력 확보와 다르며, 층마다 같은 인원을 뽑으면 전체 효과의 모집단 가중치를 다시 정해야 한다.

Primary는 전체 배정 사용자를 분모로 한 30일 재방문율(ITT). Secondary는 구매율·객단가이며 주문 ID가 없어 객단가는 운영 데이터 확보 후 산식을 정한다. Guardrail은 수신거부·비용·마진이며 실행 전에 허용 악화폭을 정한다. MDE는 검정력 설계 조건으로, 자동 성공 기준은 아니다. SRM·중복 배정·관찰 누락을 확인하고 임의 조기 중단을 피한다.

실제 실험을 집행하지 않았으며 수신 동의·비용 정보가 없어 바로 집행 가능한 운영 명세는 아니다. 자세한 산식과 제한은 [A/B 설계서](../docs/ab_test_design.md)를 따른다.

In [3]:
import json
audit = json.loads((config.REPORTS_DIR / "ab_design_validation.json").read_text())
print("기준일:", audit["reference_date"], audit["timezone"])
print("사전 입력 종료(미포함):", audit["feature_end_exclusive"])
print("미래 구매 이력 때문에 잘못 분류됐던 위험군:", f"{audit['reclassified_future_only_buyers']:,}명")
print("그중 4월 재방문자:", f"{audit['reclassified_returned_users']:,}명")

기준일: 2020-03-31 UTC
사전 입력 종료(미포함): 2020-04-01T00:00:00Z
미래 구매 이력 때문에 잘못 분류됐던 위험군: 32,583명
그중 4월 재방문자: 32,583명
